In [ ]:
import os
from math import sqrt

import pandas as pd
from Bio.PDB.MMCIFParser import MMCIFParser


folder_path = '/path/to/CIF/files/Cu2'

cif_files = [
    os.path.join(folder_path, file)
    for file in os.listdir(folder_path)
    if file.lower().endswith('.cif')
]

print(f"Total CIF files found: {len(cif_files)}")


def calculate_distance(atom1, atom2):
    return sqrt(
        (atom1.coord[0] - atom2.coord[0]) ** 2 +
        (atom1.coord[1] - atom2.coord[1]) ** 2 +
        (atom1.coord[2] - atom2.coord[2]) ** 2
    )


def process_cif_file(
    cif_path,
    copper_copper_distance_criteria=5.0,
    copper_other_distance_criteria_3=3.0
):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure('cif_structure', cif_path)
    first_model = next(structure.get_models())

    copper_atoms = []
    other_atoms = []

    for atom in first_model.get_atoms():
        residue = atom.get_parent()
        resname = residue.get_resname().strip().upper()

        if atom.element == 'CU' and resname == 'CU':
            copper_atoms.append(atom)
        else:
            other_atoms.append(atom)

    copper_copper_interactions = []
    copper_other_interactions_3A = []

    for i, copper_atom_1 in enumerate(copper_atoms):
        copper_chain_id_1 = copper_atom_1.get_parent().get_parent().id
        copper_residue_id_1 = copper_atom_1.get_parent().get_id()[1]

        for copper_atom_2 in copper_atoms[i + 1:]:
            distance = calculate_distance(copper_atom_1, copper_atom_2)

            if distance <= copper_copper_distance_criteria:
                copper_residue_id_2 = copper_atom_2.get_parent().get_id()[1]
                copper_chain_id_2 = copper_atom_2.get_parent().get_parent().id

                copper_copper_interactions.append({
                    'PDB_ID': os.path.basename(cif_path).split('.')[0],
                    'Copper_ID_1': copper_residue_id_1,
                    'Copper_Chain_ID_1': copper_chain_id_1,
                    'Copper_ID_2': copper_residue_id_2,
                    'Copper_Chain_ID_2': copper_chain_id_2,
                    'Distance': distance
                })

    for copper_atom in copper_atoms:
        if all(
            calculate_distance(copper_atom, other_cu) > copper_copper_distance_criteria
            for other_cu in copper_atoms
            if copper_atom != other_cu
        ):
            copper_chain_id = copper_atom.get_parent().get_parent().id
            copper_residue_id = copper_atom.get_parent().get_id()[1]
            copper_serial = copper_atom.serial_number

            for atom in other_atoms:
                distance = calculate_distance(copper_atom, atom)

                if distance <= copper_other_distance_criteria_3:
                    residue = atom.get_parent()
                    residue_id = residue.get_id()[1]
                    residue_name = residue.get_resname()
                    atom_name = atom.get_name()
                    atom_element = atom.element
                    atom_chain_id = atom.get_parent().get_parent().id

                    copper_other_interactions_3A.append({
                        'PDB_ID': os.path.basename(cif_path).split('.')[0],
                        'Copper_Serial': copper_serial,
                        'Copper_ID': copper_residue_id,
                        'Copper_Chain_ID': copper_chain_id,
                        'Residue_ID': residue_id,
                        'Residue_Name': residue_name,
                        'Atom_Name': atom_name,
                        'Atom_Element': atom_element,
                        'Atom_Chain_ID': atom_chain_id,
                        'Distance': distance
                    })

    return copper_copper_interactions, copper_other_interactions_3A


all_copper_copper_interactions = []
all_copper_other_interactions_3A = []

for cif_path in cif_files:
    copper_copper, copper_other_3A = process_cif_file(cif_path)

    all_copper_copper_interactions.extend(copper_copper)
    all_copper_other_interactions_3A.extend(copper_other_3A)


copper_copper_df = pd.DataFrame(all_copper_copper_interactions)
copper_other_df_3A = pd.DataFrame(all_copper_other_interactions_3A)


copper_copper_output_file = '/path/to/output/CU-Copper_Copper_Interactions.csv'
copper_other_output_file_3A = '/path/to/output/CU-Copper_Other_Interactions_3A.csv'


copper_copper_df.to_csv(copper_copper_output_file, index=False)
copper_other_df_3A.to_csv(copper_other_output_file_3A, index=False)


print(f"Copper-Copper interaction results saved to {copper_copper_output_file}")
print(f"Copper-Other (within 3 Å) interaction results saved to {copper_other_output_file_3A}")


electronegative_elements = ['O', 'N', 'S', 'F', 'Cl', 'Br', 'I']

copper_other_df_3A = pd.read_csv(copper_other_output_file_3A)

electronegative_df = copper_other_df_3A[
    copper_other_df_3A['Atom_Element'].isin(electronegative_elements)
]

copper_other_electronegative_output_file = (
    '/path/to/output/CU-Copper_Other_Interactions_3A-Electronegative.csv'
)

electronegative_df.to_csv(
    copper_other_electronegative_output_file,
    index=False
)

print(
    f"Electronegative atom interaction results saved to "
    f"{copper_other_electronegative_output_file}"
)

In [ ]:
import os
from math import sqrt

import pandas as pd
from Bio.PDB.MMCIFParser import MMCIFParser


folder_path = '/path/to/CIF/files/CU1'

cif_files = [
    os.path.join(folder_path, file)
    for file in os.listdir(folder_path)
    if file.lower().endswith('.cif')
]

print(f"Total CIF files found: {len(cif_files)}")


def calculate_distance(atom1, atom2):
    return sqrt(
        (atom1.coord[0] - atom2.coord[0]) ** 2 +
        (atom1.coord[1] - atom2.coord[1]) ** 2 +
        (atom1.coord[2] - atom2.coord[2]) ** 2
    )


def process_cif_file(
    cif_path,
    copper_copper_distance_criteria=5.0,
    copper_other_distance_criteria_3=3.0
):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure('cif_structure', cif_path)
    first_model = next(structure.get_models())

    copper_atoms = []
    other_atoms = []

    for atom in first_model.get_atoms():
        residue = atom.get_parent()

        if atom.element == 'CU' and residue.get_resname().strip() == 'CU1':
            copper_atoms.append(atom)
        else:
            other_atoms.append(atom)

    copper_copper_interactions = []
    copper_other_interactions_3A = []

    for i, copper_atom_1 in enumerate(copper_atoms):
        copper_chain_id_1 = copper_atom_1.get_parent().get_parent().id
        copper_residue_id_1 = copper_atom_1.get_parent().get_id()[1]

        for copper_atom_2 in copper_atoms[i + 1:]:
            distance = calculate_distance(copper_atom_1, copper_atom_2)

            if distance <= copper_copper_distance_criteria:
                copper_residue_id_2 = copper_atom_2.get_parent().get_id()[1]
                copper_chain_id_2 = copper_atom_2.get_parent().get_parent().id

                copper_copper_interactions.append({
                    'PDB_ID': os.path.basename(cif_path).split('.')[0],
                    'Copper_ID_1': copper_residue_id_1,
                    'Copper_Chain_ID_1': copper_chain_id_1,
                    'Copper_ID_2': copper_residue_id_2,
                    'Copper_Chain_ID_2': copper_chain_id_2,
                    'Distance': distance
                })

    for copper_atom in copper_atoms:
        if all(
            calculate_distance(copper_atom, other_cu) > copper_copper_distance_criteria
            for other_cu in copper_atoms
            if copper_atom != other_cu
        ):
            copper_chain_id = copper_atom.get_parent().get_parent().id
            copper_residue_id = copper_atom.get_parent().get_id()[1]
            copper_serial = copper_atom.serial_number

            for atom in other_atoms:
                distance = calculate_distance(copper_atom, atom)

                if distance <= copper_other_distance_criteria_3:
                    residue = atom.get_parent()
                    residue_id = residue.get_id()[1]
                    residue_name = residue.get_resname()
                    atom_name = atom.get_name()
                    atom_element = atom.element
                    atom_chain_id = atom.get_parent().get_parent().id

                    copper_other_interactions_3A.append({
                        'PDB_ID': os.path.basename(cif_path).split('.')[0],
                        'Copper_Serial': copper_serial,
                        'Copper_ID': copper_residue_id,
                        'Copper_Chain_ID': copper_chain_id,
                        'Residue_ID': residue_id,
                        'Residue_Name': residue_name,
                        'Atom_Name': atom_name,
                        'Atom_Element': atom_element,
                        'Atom_Chain_ID': atom_chain_id,
                        'Distance': distance
                    })

    return copper_copper_interactions, copper_other_interactions_3A


all_copper_copper_interactions = []
all_copper_other_interactions_3A = []

for cif_path in cif_files:
    copper_copper, copper_other_3A = process_cif_file(cif_path)

    all_copper_copper_interactions.extend(copper_copper)
    all_copper_other_interactions_3A.extend(copper_other_3A)


copper_copper_df = pd.DataFrame(all_copper_copper_interactions)
copper_other_df_3A = pd.DataFrame(all_copper_other_interactions_3A)


copper_copper_output_file = '/path/to/output/CU1-Copper_Copper_Interactions.csv'
copper_other_output_file_3A = '/path/to/output/CU1-Copper_Other_Interactions_3A.csv'


copper_copper_df.to_csv(copper_copper_output_file, index=False)
copper_other_df_3A.to_csv(copper_other_output_file_3A, index=False)


print(f"Copper-Copper interaction results saved to {copper_copper_output_file}")
print(f"Copper-Other (within 3 Å) interaction results saved to {copper_other_output_file_3A}")


electronegative_elements = ['O', 'N', 'S', 'F', 'Cl', 'Br', 'I']

copper_other_df_3A = pd.read_csv(copper_other_output_file_3A)

electronegative_df = copper_other_df_3A[
    copper_other_df_3A['Atom_Element'].isin(electronegative_elements)
]


copper_other_electronegative_output_file = (
    '/path/to/output/CU1-Copper_Other_Interactions_3A-Electronegative.csv'
)

electronegative_df.to_csv(
    copper_other_electronegative_output_file,
    index=False
)

print(
    f"Electronegative atom interaction results saved to "
    f"{copper_other_electronegative_output_file}"
)

In [ ]:
import pandas as pd


input_files = {
    "CU": "CU-Copper_Other_Interactions_3A-Electronegative.csv",
    "CU1": "CU1-Copper_Other_Interactions_3A-Electronegative.csv"
}


allowed_residues = {
    'HOH', 'OH', 'O',
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS',
    'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR',
    'TRP', 'TYR', 'VAL'
}


def process_file(input_file, output_file, cu_type):

    dfi = pd.read_csv(input_file)

    dfi['PDB_Copper_Chain'] = (
        dfi['PDB_ID'] + "_" +
        dfi['Copper_ID'].astype(str) + "_" +
        dfi['Copper_Chain_ID']
    )

    allowed_sites = dfi.groupby('PDB_Copper_Chain').filter(
        lambda x: set(x['Residue_Name']).issubset(allowed_residues)
    )

    max_ligands = (
        allowed_sites
        .groupby('PDB_Copper_Chain')
        .size()
        .max()
    )

    columns = (
        ['PDB_Copper_Chain'] +
        [f'Residue_{i+1}' for i in range(max_ligands)] +
        [f'Atom_{i+1}' for i in range(max_ligands)] +
        [f'Distance_{i+1}' for i in range(max_ligands)] +
        [f'Atom_Chain_ID_{i+1}' for i in range(max_ligands)]
    )

    result_rows = []

    for pdb_copper_chain, group in allowed_sites.groupby('PDB_Copper_Chain'):
        group = group.sort_values('Distance').reset_index(drop=True)

        residues = []
        atoms = []
        distances = []
        atom_chain_ids = []

        for _, row in group.iterrows():
            residues.append(
                f"{row['Residue_Name']}_{row['Residue_ID']}_{row['Atom_Chain_ID']}"
            )
            atoms.append(row['Atom_Name'])
            distances.append(row['Distance'])
            atom_chain_ids.append(row['Atom_Chain_ID'])

        pad_len = max_ligands - len(residues)

        residues += [None] * pad_len
        atoms += [None] * pad_len
        distances += [None] * pad_len
        atom_chain_ids += [None] * pad_len

        result_rows.append(
            [pdb_copper_chain] +
            residues[:max_ligands] +
            atoms[:max_ligands] +
            distances[:max_ligands] +
            atom_chain_ids[:max_ligands]
        )

    df = pd.DataFrame(result_rows, columns=columns)

    res_cols = [f"Residue_{i}" for i in range(1, 9)]
    atom_cols = [f"Atom_{i}" for i in range(1, 9)]

    def analyze_site(row):
        residue_atom_map = {}
        total_atoms = 0

        for r_col, a_col in zip(res_cols, atom_cols):
            residue = row[r_col]
            atom = row[a_col]

            if pd.notna(residue) and pd.notna(atom):
                total_atoms += 1
                residue_atom_map.setdefault(residue, set()).add(atom)

        residue_denticities = {
            res: len(atoms)
            for res, atoms in residue_atom_map.items()
        }

        max_residue_denticity = max(
            residue_denticities.values(),
            default=0
        )

        if max_residue_denticity >= 4:
            site_denticity = "Polydentate"
        elif max_residue_denticity == 3:
            site_denticity = "Tridentate"
        elif max_residue_denticity == 2:
            site_denticity = "Bidentate"
        else:
            site_denticity = "Monodentate"

        return pd.Series({
            "Ligand_Count": total_atoms,
            "Max_Residue_Denticity": max_residue_denticity,
            "Site_Denticity": site_denticity
        })

    df[
        ["Ligand_Count", "Max_Residue_Denticity", "Site_Denticity"]
    ] = df.apply(analyze_site, axis=1)

    residue_cols = [f"Residue_{i}" for i in range(1, 9)]

    def count_unique_residues(row):
        seen = set()

        for i, res_col in enumerate(residue_cols, start=1):
            residue = row[res_col]
            chain = row[f"Atom_Chain_ID_{i}"]

            if pd.isna(residue):
                continue

            residue_str = str(residue).upper().replace("-", "_")
            parts = residue_str.split("_")

            if len(parts) >= 2:
                if pd.notna(chain):
                    unique_id = f"{parts[0]}_{parts[1]}_{chain}"
                else:
                    unique_id = f"{parts[0]}_{parts[1]}"
            else:
                unique_id = parts[0]

            seen.add(unique_id)

        return len(seen)

    max_unique_res = df.apply(
        count_unique_residues,
        axis=1
    ).max()

    new_res_cols = [
        f"res_{i+1}"
        for i in range(max_unique_res)
    ]

    def extract_unique_residues(row):
        seen = set()
        unique_res = []

        for i, res_col in enumerate(residue_cols, start=1):
            residue = row[res_col]
            chain = row[f"Atom_Chain_ID_{i}"]

            if pd.isna(residue):
                continue

            residue_str = str(residue).upper().replace("-", "_")
            parts = residue_str.split("_")

            if len(parts) >= 2:
                if pd.notna(chain):
                    unique_id = f"{parts[0]}_{parts[1]}_{chain}"
                else:
                    unique_id = f"{parts[0]}_{parts[1]}"
            else:
                unique_id = parts[0]

            if unique_id not in seen:
                seen.add(unique_id)
                unique_res.append(unique_id)

        unique_res += [None] * (
            max_unique_res - len(unique_res)
        )

        return pd.Series(unique_res)

    df[new_res_cols] = df.apply(
        extract_unique_residues,
        axis=1
    )

    def drop_residue_id(val):
        if not isinstance(val, str):
            return val
        return val.split("_")[0]

    df[new_res_cols] = df[new_res_cols].applymap(
        drop_residue_id
    )

    df["CU_Type"] = cu_type

    df = df[
        df["Ligand_Count"].isin([2, 3, 4, 5, 6])
    ]

    df.to_csv(output_file, index=False)

    print(f"Processed: {input_file}")
    print(f"CU_Type: {cu_type}")
    print(f"Rows retained: {len(df)}")
    print(f"Saved: {output_file}")


for metal_type, input_file in input_files.items():

    if metal_type == "CU":
        cu_type = "Cu1"
    else:
        cu_type = "Cu2"

    output_file = (
        f"{metal_type}-Copper_Other_Interactions_3A-Electronegative-processed.csv"
    )

    process_file(
        input_file,
        output_file,
        cu_type
    )